In [1]:
from bs4 import BeautifulSoup
print("BeautifulSoup installed!")


ModuleNotFoundError: No module named 'bs4'

In [3]:
import pandas as pd

In [4]:
import sys
print("Kernel Python:", sys.executable)

!{sys.executable} -m pip install beautifulsoup4 --upgrade


Kernel Python: /anaconda/envs/azureml_py310_sdkv2/bin/python
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [beautifulsoup4]


In [5]:
from bs4 import BeautifulSoup
print("BeautifulSoup installed and working!")

BeautifulSoup installed and working!


In [7]:
import os
from bs4 import BeautifulSoup
import pandas as pd

def extract_info(html_file):
    with open(html_file, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")

    # ---- Extract fields with fallbacks ----
    def sel(*queries):
        for q in queries:
            el = soup.select_one(q)
            if el:
                return el.get_text(strip=True)
        return ""

    job_title = sel("h1", "h2", "[data-testid='jobTitle']")
    company    = sel("[data-company]", ".jobsearch-CompanyInfoWithoutHeaderImage")
    location   = sel("[data-testid='text-location']", ".jobsearch-CompanyLocation", ".css-6z8o9s")
    salary     = sel("[data-testid='salary']", ".attribute_snippet", ".css-2iqe2o")

    return {
        "file": html_file,
        "job_title": job_title,
        "company": company,
        "location": location,
        "salary": salary
    }

# ---- Scan all HTML files ----
records = []
for file in sorted(os.listdir(".")):
    if file.endswith(".html"):
        try:
            info = extract_info(file)
            records.append(info)
        except Exception as e:
            print(f"Error parsing {file}: {e}")

# ---- Build DataFrame ----
df = pd.DataFrame(records)

# ---- Export to HTML ----
# df.to_html("output.html", index=False)

print("✔ Done! Extracted data saved in output.html")

✔ Done! Extracted data saved in output.html


In [8]:
df

,file,job_title,company,location,salary
0,1.html,AI Engineer Lead,,,
1,10.html,Head of Financial and Business Intelligence,,,
2,11.html,Investment Associate – GenAI,,,
3,12.html,"Manager, Quantitative Analysis - Model Risk Of...",,,
4,13.html,"Managing Director, Lead",,,
5,14.html,"Principal Applied Scientist, AWS Agentic AI Sc...",,,
6,15.html,"Product Manager – AI & Data Platforms, Financi...",,,
7,16.html,Real Assets Digital and AI - Digital Solutions...,,,
8,17.html,"Senior Manager - Finance AI & Data Strategy, B...",,,
9,18.html,Senior Manager - Finance AI & Data Strategy (C...,,,


In [9]:
import os
from bs4 import BeautifulSoup
import pandas as pd
import re

def extract_text(soup, selectors):
    """Try many selectors; return first match text."""
    for s in selectors:
        el = soup.select_one(s)
        if el:
            return el.get_text(" ", strip=True)
    return ""

def extract_many(soup, selectors):
    """Return joined text for blocks with multiple nodes."""
    items = []
    for s in selectors:
        for el in soup.select(s):
            txt = el.get_text(" ", strip=True)
            if txt:
                items.append(txt)
    return "\n".join(items).strip()

def extract_info(html_file):
    with open(html_file, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")

    # -------- BASIC FIELDS --------
    job_title = extract_text(soup, [
        "h1", "h2",
        "[data-testid='jobTitle']",
        ".jobsearch-JobInfoHeader-title"
    ])

    company = extract_text(soup, [
        ".css-1cjkto6",                        # Indeed company header
        "[data-company]",
        ".jobsearch-CompanyInfoWithoutHeaderImage",
        ".css-1h7lukc"
    ])

    location = extract_text(soup, [
        "[data-testid='text-location']",
        ".jobsearch-CompanyLocation",
        ".css-6z8o9s"
    ])

    salary = extract_text(soup, [
        "[data-testid='salary']", 
        ".attribute_snippet",
        ".css-2iqe2o",
        ".salary-snippet-container",
        ".jobsearch-JobMetadataHeader-item"
    ])

    # -------- JOB DESCRIPTION --------
    job_description = extract_many(soup, [
        "#jobDescriptionText",
        "[data-testid='jobDescription']",
        ".jobsearch-jobDescriptionText",
        ".css-1kzrs93"
    ])

    # -------- RESPONSIBILITIES & HIGHLIGHTS --------
    responsibilities = extract_many(soup, [
        "ul", "ol",
        ".css-1sb02f4",  # bullet sections
        ".css-1p0sjhy"
    ])

    # -------- JOB DETAILS BOX (Job Type, Schedule, Pay, etc.) --------
    details_labels = soup.select(".jobsearch-JobDescriptionSection-sectionItemKey")
    details_values = soup.select(".jobsearch-JobDescriptionSection-sectionItemValue")

    details = {}
    for lbl, val in zip(details_labels, details_values):
        details[lbl.get_text(strip=True)] = val.get_text(" ", strip=True)

    # Flatten details
    for k, v in details.items():
        details[k.lower().replace(" ", "_")] = v

    # -------- POSTED DATE --------
    posted = extract_text(soup, [
        ".css-qvloho",
        ".jobsearch-JobMetadataFooter",
        "[data-testid='text-date']"
    ])

    return {
        "file": html_file,
        "job_title": job_title,
        "company": company,
        "location": location,
        "salary": salary,
        "posted": posted,
        "job_description": job_description,
        "responsibilities": responsibilities,
        **details
    }

# -------- MAIN LOOP --------
rows = []
for file in sorted(os.listdir(".")):
    if file.endswith(".html"):
        print("Parsing:", file)
        data = extract_info(file)
        rows.append(data)

df = pd.DataFrame(rows)

# Save consolidated HTML
# df.to_html("output_full.html", index=False)

print("✔ Completed. Saved full extraction to output_full.html")

Parsing: 1.html
Parsing: 10.html
Parsing: 11.html
Parsing: 12.html
Parsing: 13.html
Parsing: 14.html
Parsing: 15.html
Parsing: 16.html
Parsing: 17.html
Parsing: 18.html
Parsing: 19.html
Parsing: 2.html
Parsing: 20.html
Parsing: 21.html
Parsing: 22.html
Parsing: 23.html
Parsing: 24.html
Parsing: 25.html
Parsing: 3.html
Parsing: 4.html
Parsing: 5.html
Parsing: 6.html
Parsing: 7.html
Parsing: 8.html
Parsing: 9.html
Parsing: output.html
✔ Completed. Saved full extraction to output_full.html


In [10]:
df

,file,job_title,company,location,salary,posted,job_description,responsibilities
0,1.html,AI Engineer Lead,,,,Report job Apply on company site save-icon,Job Description What is the opportunity? Join ...,Home Company reviews Find salaries\nMessages U...
1,10.html,Head of Financial and Business Intelligence,,,,Report job Apply on company site save-icon &nbsp;,WisdomTree is seeking a Head of Financial and ...,Home Company reviews Find salaries\nMessages U...
2,11.html,Investment Associate – GenAI,,,,If you require alternative methods of applicat...,Investment Associate – GenAI Business Strategy...,Home Company reviews Find salaries\nMessages U...
3,12.html,"Manager, Quantitative Analysis - Model Risk Of...",,,,Report job Apply on company site save-icon &nbsp;,"Posted Date 3/19/2026 Description Manager, Qua...",Home Company reviews Find salaries\nMessages U...
4,13.html,"Managing Director, Lead",,,,If you require alternative methods of applicat...,About the Company We are seeking a visionary a...,Home Company reviews Find salaries\nMessages U...
5,14.html,"Principal Applied Scientist, AWS Agentic AI Sc...",,,,Report job Apply on company site save-icon &nbsp;,DESCRIPTION AWS Agentic AI is looking for a wo...,Home Company reviews Find salaries\nMessages U...
6,15.html,"Product Manager – AI & Data Platforms, Financi...",,,,Report job Apply on company site save-icon &nbsp;,"Location: New York, New York, United States Ab...",Home Company reviews Find salaries\nMessages U...
7,16.html,Real Assets Digital and AI - Digital Solutions...,,,,Report job Apply on company site save-icon &nbsp;,Transform how real assets performance is monit...,Home Company reviews Find salaries\nMessages U...
8,17.html,"Senior Manager - Finance AI & Data Strategy, B...",,,,Report job Apply on company site save-icon,Accenture is a leading global professional ser...,Home Company reviews Find salaries\nMessages U...
9,18.html,Senior Manager - Finance AI & Data Strategy (C...,,,,Report job Apply on company site save-icon &nbsp;,Enterprise AI Value Strategy Senior Manager | ...,Home Company reviews Find salaries\nMessages U...


In [11]:
df.columns

Index(['file', 'job_title', 'company', 'location', 'salary', 'posted',
       'job_description', 'responsibilities'],
      dtype='object')

In [12]:
df.salary

0     
1     
2     
3     
4     
5     
6     
7     
8     
9     
10    
11    
12    
13    
14    
15    
16    
17    
18    
19    
20    
21    
22    
23    
24    
25    
Name: salary, dtype: object

In [13]:
import os
from bs4 import BeautifulSoup
import pandas as pd
import re

# -------------------------------------------
# Helper functions
# -------------------------------------------
def extract_text(soup, selectors):
    """Return first matching text from selectors."""
    for s in selectors:
        el = soup.select_one(s)
        if el:
            return el.get_text(" ", strip=True)
    return ""

def extract_many(soup, selectors):
    """Return joined text from many nodes."""
    items = []
    for s in selectors:
        for el in soup.select(s):
            txt = el.get_text(" ", strip=True)
            if txt:
                items.append(txt)
    return "\n".join(items).strip()


# -------------------------------------------
# Extract info from each HTML file
# -------------------------------------------
def extract_info(html_file):
    with open(html_file, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")

    # ------- JOB TITLE -------
    job_title = extract_text(soup, [
        "h1",
        "[data-testid='jobTitle']",
        ".jobsearch-JobInfoHeader-title",
    ])

    # ------- COMPANY (FIXED) -------
    # Try visible company blocks first
    company = extract_text(soup, [
        ".jobsearch-CompanyInfoWithoutHeaderImage",
        "[data-company]",
        ".css-1cjkto6",
        ".css-1h7lukc",
    ])

    # Fallback: meta tags (works for the file you uploaded)
    if not company:
        meta1 = soup.find("meta", {"property": "og:description"})
        if meta1 and meta1.get("content"):
            company = meta1["content"]

    if not company:
        meta2 = soup.find("meta", {"name": "twitter:description"})
        if meta2 and meta2.get("content"):
            company = meta2["content"]

    # ------- LOCATION -------
    location = extract_text(soup, [
        "[data-testid='text-location']",
        ".jobsearch-CompanyLocation",
        ".css-6z8o9s",
    ])

    # ------- SALARY (FIXED) -------
    salary = extract_text(soup, [
        "[data-testid='salary']",
        ".jobsearch-JobMetadataHeader-item",
        ".css-1bluz6i",
        ".attribute_snippet",
        ".css-2iqe2o",
    ])

    # Deep fallback: extract salary from JSON in scripts if needed
    if not salary:
        for s in soup.find_all("script"):
            if s.string and "salary" in s.string.lower():
                m = re.search(r"salary[^:]*:\s*\"([^\"]+)\"", s.string, re.I)
                if m:
                    salary = m.group(1)
                    break

    # ------- JOB DESCRIPTION -------
    job_description = extract_many(soup, [
        "#jobDescriptionText",
        "[data-testid='jobDescription']",
        ".jobsearch-jobDescriptionText",
        ".css-1kzrs93",
    ])

    # ------- BULLET RESPONSIBILITIES -------
    responsibilities = extract_many(soup, [
        "ul",
        "ol",
        ".css-1sb02f4",
        ".css-1p0sjhy",
    ])

    # ------- JOB DETAILS SECTION -------
    details_labels = soup.select(".jobsearch-JobDescriptionSection-sectionItemKey")
    details_values = soup.select(".jobsearch-JobDescriptionSection-sectionItemValue")

    details = {}
    for lbl, val in zip(details_labels, details_values):
        key = lbl.get_text(strip=True).lower().replace(" ", "_")
        value = val.get_text(" ", strip=True)
        details[key] = value

    # ------- POSTED DATE -------
    posted = extract_text(soup, [
        ".css-qvloho",
        ".jobsearch-JobMetadataFooter",
        "[data-testid='text-date']",
    ])

    return {
        "file": html_file,
        "job_title": job_title,
        "company": company,
        "location": location,
        "salary": salary,
        "posted": posted,
        "job_description": job_description,
        "responsibilities": responsibilities,
        **details  # merge in Job Type, Shift, etc. if available
    }


# -------------------------------------------
# MAIN LOOP
# -------------------------------------------
rows = []
for file in sorted(os.listdir(".")):
    if file.endswith(".html"):
        print("Parsing:", file)
        rows.append(extract_info(file))

df = pd.DataFrame(rows)

# Save to single HTML report
# df.to_html("output_fixed.html", index=False, escape=False)

print("✔ DONE — Saved as output_fixed.html")

Parsing: 1.html
Parsing: 10.html
Parsing: 11.html
Parsing: 12.html
Parsing: 13.html
Parsing: 14.html
Parsing: 15.html
Parsing: 16.html
Parsing: 17.html
Parsing: 18.html
Parsing: 19.html
Parsing: 2.html
Parsing: 20.html
Parsing: 21.html
Parsing: 22.html
Parsing: 23.html
Parsing: 24.html
Parsing: 25.html
Parsing: 3.html
Parsing: 4.html
Parsing: 5.html
Parsing: 6.html
Parsing: 7.html
Parsing: 8.html
Parsing: 9.html
Parsing: output.html
✔ DONE — Saved as output_fixed.html


In [14]:
df

,file,job_title,company,location,salary,posted,job_description,responsibilities
0,1.html,AI Engineer Lead,Royal Bank of Canada,,"$300,000",Report job Apply on company site save-icon,Job Description What is the opportunity? Join ...,Home Company reviews Find salaries\nMessages U...
1,10.html,Head of Financial and Business Intelligence,WisdomTree,,"Head of Finance salaries in New York, NY",Report job Apply on company site save-icon &nbsp;,WisdomTree is seeking a Head of Financial and ...,Home Company reviews Find salaries\nMessages U...
2,11.html,Investment Associate – GenAI,Elliot Partnership,,"Investment Associate salaries in New York, NY",If you require alternative methods of applicat...,Investment Associate – GenAI Business Strategy...,Home Company reviews Find salaries\nMessages U...
3,12.html,"Manager, Quantitative Analysis - Model Risk Of...",Information Technology Senior Management Forum,,Posted Date\n3\u002F19\u002F2026\nDescription\...,Report job Apply on company site save-icon &nbsp;,"Posted Date 3/19/2026 Description Manager, Qua...",Home Company reviews Find salaries\nMessages U...
4,13.html,"Managing Director, Lead",Precision AQ,,"Managing Director salaries in New York, NY",If you require alternative methods of applicat...,About the Company We are seeking a visionary a...,Home Company reviews Find salaries\nMessages U...
5,14.html,"Principal Applied Scientist, AWS Agentic AI Sc...","Amazon Development Center U.S., Inc.",,Principal Applied Scientist salaries in New Yo...,Report job Apply on company site save-icon &nbsp;,DESCRIPTION AWS Agentic AI is looking for a wo...,Home Company reviews Find salaries\nMessages U...
6,15.html,"Product Manager – AI & Data Platforms, Financi...",Turing,,"Product Manager salaries in New York, NY",Report job Apply on company site save-icon &nbsp;,"Location: New York, New York, United States Ab...",Home Company reviews Find salaries\nMessages U...
7,16.html,Real Assets Digital and AI - Digital Solutions...,Macquarie Group Limited,,Digital Contact Solutions Director salaries in...,Report job Apply on company site save-icon &nbsp;,Transform how real assets performance is monit...,Home Company reviews Find salaries\nMessages U...
8,17.html,"Senior Manager - Finance AI & Data Strategy, B...",Accenture,,"$338,300",Report job Apply on company site save-icon,Accenture is a leading global professional ser...,Home Company reviews Find salaries\nMessages U...
9,18.html,Senior Manager - Finance AI & Data Strategy (C...,Logic,,"Media Strategy Manager salaries in New York, NY",Report job Apply on company site save-icon &nbsp;,Enterprise AI Value Strategy Senior Manager | ...,Home Company reviews Find salaries\nMessages U...


In [15]:
import os
from bs4 import BeautifulSoup
import pandas as pd
import re

def extract_text(soup, selectors):
    for s in selectors:
        el = soup.select_one(s)
        if el:
            return el.get_text(" ", strip=True)
    return ""

def extract_many(soup, selectors):
    items = []
    for s in selectors:
        for el in soup.select(s):
            txt = el.get_text(" ", strip=True)
            if txt:
                items.append(txt)
    return "\n".join(items).strip()


def clean_salary(text):
    """Return text only if it looks like a REAL salary."""
    if not text:
        return ""

    t = text.lower()

    # Reject fake salary strings
    bad_patterns = [
        "salaries in",
        "find salaries",
        "salary.com",
        "popular salaries",
        "salary range for"
    ]
    if any(b in t for b in bad_patterns):
        return ""

    # Accept if real salary-like
    real_salary_patterns = [
        r"\$\d",
        r"\d+k",
        r"\d{2,3}\,\d{3}",
        r"\$\d+\/hr",
        r"\$\d+\s*-\s*\$\d+"
    ]

    for p in real_salary_patterns:
        if re.search(p, text, re.I):
            return text.strip()

    return ""


def extract_info(html_file):
    with open(html_file, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")

    # -------------------- JOB TITLE --------------------
    job_title = extract_text(soup, [
        "h1",
        "[data-testid='jobTitle']",
        ".jobsearch-JobInfoHeader-title",
    ])

    # -------------------- COMPANY --------------------
    company = extract_text(soup, [
        ".jobsearch-CompanyInfoWithoutHeaderImage",
        "[data-company]",
        ".css-1cjkto6",
        ".css-1h7lukc",
    ])

    # Fallback: meta property (your file uses this)
    if not company:
        meta1 = soup.find("meta", {"property": "og:description"})
        if meta1 and meta1.get("content"):
            company = meta1["content"]

    if not company:
        meta2 = soup.find("meta", {"name": "twitter:description"})
        if meta2 and meta2.get("content"):
            company = meta2["content"]

    # -------------------- LOCATION --------------------
    location = extract_text(soup, [
        "[data-testid='text-location']",
        ".jobsearch-CompanyLocation",
        ".css-6z8o9s",
    ])

    # -------------------- STRICT SALARY FIX --------------------
    salary_candidates = []

    # Look in true salary containers
    for sel in [
        "[data-testid='salary']",
        ".jobsearch-JobMetadataHeader-item",
        ".css-1bluz6i",
        ".css-2iqe2o",
        ".salary-snippet-container",
    ]:
        for el in soup.select(sel):
            text = el.get_text(" ", strip=True)
            s = clean_salary(text)
            if s:
                salary_candidates.append(s)

    # Deep fallback: JSON in scripts
    if not salary_candidates:
        for s in soup.find_all("script"):
            if s.string and "salary" in s.string.lower():
                m = re.search(r'"salary"\s*:\s*"([^"]+)"', s.string, re.I)
                if m:
                    ss = clean_salary(m.group(1))
                    if ss:
                        salary_candidates.append(ss)
                        break

    salary = salary_candidates[0] if salary_candidates else ""

    # -------------------- DESCRIPTION --------------------
    job_description = extract_many(soup, [
        "#jobDescriptionText",
        "[data-testid='jobDescription']",
        ".jobsearch-jobDescriptionText",
        ".css-1kzrs93",
    ])

    # -------------------- RESPONSIBILITIES --------------------
    responsibilities = extract_many(soup, [
        "ul",
        "ol",
        ".css-1sb02f4",
        ".css-1p0sjhy",
    ])

    # -------------------- DETAILS --------------------
    details = {}
    labels = soup.select(".jobsearch-JobDescriptionSection-sectionItemKey")
    values = soup.select(".jobsearch-JobDescriptionSection-sectionItemValue")

    for lbl, val in zip(labels, values):
        key = lbl.get_text(strip=True).lower().replace(" ", "_")
        value = val.get_text(" ", strip=True)
        details[key] = value

    # -------------------- POSTED DATE --------------------
    posted = extract_text(soup, [
        ".css-qvloho",
        ".jobsearch-JobMetadataFooter",
        "[data-testid='text-date']",
    ])

    return {
        "file": html_file,
        "job_title": job_title,
        "company": company,
        "location": location,
        "salary": salary,
        "posted": posted,
        "job_description": job_description,
        "responsibilities": responsibilities,
        **details
    }


# -------------------- MAIN LOOP --------------------
rows = []
for file in sorted(os.listdir(".")):
    if file.endswith(".html"):
        print("Parsing:", file)
        rows.append(extract_info(file))

df = pd.DataFrame(rows)

print("✔ DONE — Salary + Company fixed. Saved as output_fixed_salary.html")


Parsing: 1.html
Parsing: 10.html
Parsing: 11.html
Parsing: 12.html
Parsing: 13.html
Parsing: 14.html
Parsing: 15.html
Parsing: 16.html
Parsing: 17.html
Parsing: 18.html
Parsing: 19.html
Parsing: 2.html
Parsing: 20.html
Parsing: 21.html
Parsing: 22.html
Parsing: 23.html
Parsing: 24.html
Parsing: 25.html
Parsing: 3.html
Parsing: 4.html
Parsing: 5.html
Parsing: 6.html
Parsing: 7.html
Parsing: 8.html
Parsing: 9.html
Parsing: output.html
✔ DONE — Salary + Company fixed. Saved as output_fixed_salary.html


In [16]:
df

,file,job_title,company,location,salary,posted,job_description,responsibilities
0,1.html,AI Engineer Lead,Royal Bank of Canada,,,Report job Apply on company site save-icon,Job Description What is the opportunity? Join ...,Home Company reviews Find salaries\nMessages U...
1,10.html,Head of Financial and Business Intelligence,WisdomTree,,,Report job Apply on company site save-icon &nbsp;,WisdomTree is seeking a Head of Financial and ...,Home Company reviews Find salaries\nMessages U...
2,11.html,Investment Associate – GenAI,Elliot Partnership,,,If you require alternative methods of applicat...,Investment Associate – GenAI Business Strategy...,Home Company reviews Find salaries\nMessages U...
3,12.html,"Manager, Quantitative Analysis - Model Risk Of...",Information Technology Senior Management Forum,,,Report job Apply on company site save-icon &nbsp;,"Posted Date 3/19/2026 Description Manager, Qua...",Home Company reviews Find salaries\nMessages U...
4,13.html,"Managing Director, Lead",Precision AQ,,,If you require alternative methods of applicat...,About the Company We are seeking a visionary a...,Home Company reviews Find salaries\nMessages U...
5,14.html,"Principal Applied Scientist, AWS Agentic AI Sc...","Amazon Development Center U.S., Inc.",,,Report job Apply on company site save-icon &nbsp;,DESCRIPTION AWS Agentic AI is looking for a wo...,Home Company reviews Find salaries\nMessages U...
6,15.html,"Product Manager – AI & Data Platforms, Financi...",Turing,,,Report job Apply on company site save-icon &nbsp;,"Location: New York, New York, United States Ab...",Home Company reviews Find salaries\nMessages U...
7,16.html,Real Assets Digital and AI - Digital Solutions...,Macquarie Group Limited,,,Report job Apply on company site save-icon &nbsp;,Transform how real assets performance is monit...,Home Company reviews Find salaries\nMessages U...
8,17.html,"Senior Manager - Finance AI & Data Strategy, B...",Accenture,,,Report job Apply on company site save-icon,Accenture is a leading global professional ser...,Home Company reviews Find salaries\nMessages U...
9,18.html,Senior Manager - Finance AI & Data Strategy (C...,Logic,,,Report job Apply on company site save-icon &nbsp;,Enterprise AI Value Strategy Senior Manager | ...,Home Company reviews Find salaries\nMessages U...


In [17]:
import os
from bs4 import BeautifulSoup
import pandas as pd
import re

# -------------------------------
# Helper: strong salary cleaner
# -------------------------------
def clean_salary(text):
    if not text:
        return ""

    t = text.lower()

    # Reject bad/irrelevant salary text
    BAD = [
        "salaries in",
        "find salaries",
        "report job",
        "apply on company site",
        "save-icon",
        "messages",
        "company reviews"
    ]
    if any(b in t for b in BAD):
        return ""

    # Real salary-like patterns
    GOOD = [
        r"\$\d",            # $160,000
        r"\d+k",            # 150k
        r"\$?\d+\/hr",      # $80/hr
        r"\$?\d+\s*-\s*\$?\d+",  # 120k - 180k
    ]

    for p in GOOD:
        if re.search(p, text, re.I):
            return text.strip()

    return ""

# -------------------------------
# Helper: extract text from selectors
# -------------------------------
def extract_text(soup, selectors):
    for s in selectors:
        el = soup.select_one(s)
        if el:
            return el.get_text(" ", strip=True)
    return ""

# -------------------------------
# Extract responsibilities ONLY inside job description
# -------------------------------
def extract_responsibilities(soup):
    container = None

    # Detect the job description wrapper
    for sel in [
        "#jobDescriptionText",
        "[data-testid='jobDescription']",
        ".jobsearch-jobDescriptionText"
    ]:
        c = soup.select_one(sel)
        if c:
            container = c
            break

    if not container:
        return ""

    # Extract only bullet items inside the description
    bullets = []
    for ul in container.find_all(["ul", "ol"]):
        for li in ul.find_all("li"):
            txt = li.get_text(" ", strip=True)
            if txt:
                bullets.append(txt)

    return "\n".join(bullets).strip()

# -------------------------------
# Extract salary from valid containers only
# -------------------------------
def extract_salary(soup):
    salary_candidates = []

    SALARY_SELECTORS = [
        "[data-testid='salary']",
        ".salary-snippet-container",
        ".jobsearch-JobMetadataHeader-item",
        ".css-1bluz6i",
    ]

    for sel in SALARY_SELECTORS:
        for el in soup.select(sel):
            cleaned = clean_salary(el.get_text(" ", strip=True))
            if cleaned:
                salary_candidates.append(cleaned)

    # Deep fallback: script JSON
    if not salary_candidates:
        for sc in soup.find_all("script"):
            if sc.string and "salary" in sc.string.lower():
                m = re.search(r'"salary"\s*:\s*"([^"]+)"', sc.string)
                if m:
                    cleaned = clean_salary(m.group(1))
                    if cleaned:
                        salary_candidates.append(cleaned)
                        break

    return salary_candidates[0] if salary_candidates else ""

# -------------------------------
# MAIN extraction
# -------------------------------
def extract_info(html_file):
    with open(html_file, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")

    job_title = extract_text(soup, [
        "h1", "[data-testid='jobTitle']", ".jobsearch-JobInfoHeader-title"
    ])

    company = extract_text(soup, [
        ".jobsearch-CompanyInfoWithoutHeaderImage",
        "[data-company]", ".css-1cjkto6", ".css-1h7lukc"
    ])

    # Meta fallback
    if not company:
        for mt in [
            ("meta", {"property": "og:description"}),
            ("meta", {"name": "twitter:description"})
        ]:
            tag = soup.find(*mt)
            if tag and tag.get("content"):
                company = tag["content"]
                break

    location = extract_text(soup, [
        "[data-testid='text-location']",
        ".jobsearch-CompanyLocation",
        ".css-6z8o9s"
    ])

    # FIXED salary
    salary = extract_salary(soup)

    posted = extract_text(soup, [
        ".css-qvloho", ".jobsearch-JobMetadataFooter",
        "[data-testid='text-date']"
    ])

    # Description
    job_description = extract_text(soup, [
        "#jobDescriptionText", "[data-testid='jobDescription']",
        ".jobsearch-jobDescriptionText"
    ])

    # FIXED responsibilities
    responsibilities = extract_responsibilities(soup)

    # Job Detail Pairs (unchanged)
    details = {}
    keys = soup.select(".jobsearch-JobDescriptionSection-sectionItemKey")
    vals = soup.select(".jobsearch-JobDescriptionSection-sectionItemValue")

    for k, v in zip(keys, vals):
        key = k.get_text(strip=True).lower().replace(" ", "_")
        details[key] = v.get_text(" ", strip=True)

    return {
        "file": html_file,
        "job_title": job_title,
        "company": company,
        "location": location,
        "salary": salary,
        "posted": posted,
        "job_description": job_description,
        "responsibilities": responsibilities,
        **details
    }

# -------------------------------
# LOOP all HTML files
# -------------------------------
rows = []
for file in sorted(os.listdir(".")):
    if file.endswith(".html"):
        print("Parsing:", file)
        rows.append(extract_info(file))

df = pd.DataFrame(rows)

print("✔ DONE — Salary & Responsibilities fixed. Output: output_final.html")

Parsing: 1.html
Parsing: 10.html
Parsing: 11.html
Parsing: 12.html
Parsing: 13.html
Parsing: 14.html
Parsing: 15.html
Parsing: 16.html
Parsing: 17.html
Parsing: 18.html
Parsing: 19.html
Parsing: 2.html
Parsing: 20.html
Parsing: 21.html
Parsing: 22.html
Parsing: 23.html
Parsing: 24.html
Parsing: 25.html
Parsing: 3.html
Parsing: 4.html
Parsing: 5.html
Parsing: 6.html
Parsing: 7.html
Parsing: 8.html
Parsing: 9.html
Parsing: output.html
✔ DONE — Salary & Responsibilities fixed. Output: output_final.html


In [18]:
df

,file,job_title,company,location,salary,posted,job_description,responsibilities
0,1.html,AI Engineer Lead,Royal Bank of Canada,,,Report job Apply on company site save-icon,Job Description What is the opportunity? Join ...,Strategic Leadership : Define and execute the ...
1,10.html,Head of Financial and Business Intelligence,WisdomTree,,,Report job Apply on company site save-icon &nbsp;,WisdomTree is seeking a Head of Financial and ...,Set the vision and lead the Financial and Busi...
2,11.html,Investment Associate – GenAI,Elliot Partnership,,,If you require alternative methods of applicat...,Investment Associate – GenAI Business Strategy...,Investment Associate – GenAI Business Strategy...
3,12.html,"Manager, Quantitative Analysis - Model Risk Of...",Information Technology Senior Management Forum,,,Report job Apply on company site save-icon &nbsp;,"Posted Date 3/19/2026 Description Manager, Qua...",Remain on the leading edge of analytical techn...
4,13.html,"Managing Director, Lead",Precision AQ,,,If you require alternative methods of applicat...,About the Company We are seeking a visionary a...,Define and execute the go-to-market strategy f...
5,14.html,"Principal Applied Scientist, AWS Agentic AI Sc...","Amazon Development Center U.S., Inc.",,,Report job Apply on company site save-icon &nbsp;,DESCRIPTION AWS Agentic AI is looking for a wo...,Develop technical breakthroughs in agentic inf...
6,15.html,"Product Manager – AI & Data Platforms, Financi...",Turing,,,Report job Apply on company site save-icon &nbsp;,"Location: New York, New York, United States Ab...",AI & Data Platform Development: Lead the end-t...
7,16.html,Real Assets Digital and AI - Digital Solutions...,Macquarie Group Limited,,,Report job Apply on company site save-icon &nbsp;,Transform how real assets performance is monit...,Proven track record leading cross-functional t...
8,17.html,"Senior Manager - Finance AI & Data Strategy, B...",Accenture,,,Report job Apply on company site save-icon,Accenture is a leading global professional ser...,Lead Finance-specific AI strategy engagements ...
9,18.html,Senior Manager - Finance AI & Data Strategy (C...,Logic,,,Report job Apply on company site save-icon &nbsp;,Enterprise AI Value Strategy Senior Manager | ...,Lead Finance-specific AI strategy engagements ...


In [19]:
df.to_csv("output_final.csv", index=False)